# Chapter 7 — RL for Recommendation Systems## DQN-Based Movie Recommender[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**GPU optional. Runtime: ~25 minutes.**This notebook builds a sequential recommendation agent that learns to suggest movies based on user history, using Q-Learning over a simulated user environment.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import random
from collections import defaultdict, deque
import matplotlib.pyplot as plt
print('Ready')

## 7.1 The Recommendation MDP- **State**: user's last 3 viewed genres + rating history- **Action**: which genre to recommend next (10 genres)- **Reward**: +1 if user watches, +3 if user rates ≥4 stars, -0.5 if user skips

In [ ]:
# Simulated User Environment
np.random.seed(42)
GENRES = ['Action','Comedy','Drama','Sci-Fi','Horror','Romance','Thriller','Documentary','Animation','Fantasy']
N_GENRES = len(GENRES)

class UserEnvironment:
    """Simulated user with genre preferences that evolve over time."""
    def __init__(self):
        self.preferences = np.random.dirichlet(np.ones(N_GENRES) * 2)
        self.history = []
        self.step_count = 0
    
    def reset(self):
        self.preferences = np.random.dirichlet(np.ones(N_GENRES) * 2)
        self.history = []
        self.step_count = 0
        state = np.zeros(N_GENRES + 3)  # preference signal + last 3 genres
        return state
    
    def step(self, recommended_genre):
        # Probability of engagement based on genre match to preferences
        p_watch = self.preferences[recommended_genre]
        watched = np.random.random() < p_watch
        
        if watched:
            rating = np.random.choice([3, 4, 5], p=[0.2, 0.5, 0.3])
            reward = 3.0 if rating >= 4 else 1.0
        else:
            reward = -0.5
        
        self.history.append(recommended_genre)
        self.step_count += 1
        done = self.step_count >= 20
        
        # State: genre preferences (partially observed) + last 3 recommendations
        state = np.zeros(N_GENRES + 3)
        state[:N_GENRES] = self.preferences + np.random.normal(0, 0.05, N_GENRES)
        for i, g in enumerate(self.history[-3:]):
            state[N_GENRES + i] = g / N_GENRES
        
        return state, reward, done

env = UserEnvironment()
state = env.reset()
print(f'State shape: {state.shape}')
print(f'Sample state: {state.round(3)}')

## 7.2 DQN Recommender Agent

In [ ]:
class RecDQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_GENRES + 3, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, N_GENRES)
        )
    def forward(self, x): return self.net(x)

agent = RecDQN()
target_net = RecDQN()
target_net.load_state_dict(agent.state_dict())
optimizer = torch.optim.Adam(agent.parameters(), lr=1e-3)
buffer = deque(maxlen=5000)

EPSILON = 0.5
rewards_log = []

for episode in range(500):
    env = UserEnvironment()
    state = env.reset()
    ep_reward = 0
    while True:
        if random.random() < EPSILON:
            action = random.randint(0, N_GENRES-1)
        else:
            with torch.no_grad():
                action = agent(torch.FloatTensor(state)).argmax().item()
        next_state, reward, done = env.step(action)
        buffer.append((state, action, reward, next_state, done))
        ep_reward += reward
        state = next_state
        if len(buffer) >= 64:
            batch = random.sample(buffer, 64)
            s,a,r,ns,d = zip(*batch)
            s_t = torch.FloatTensor(s)
            a_t = torch.LongTensor(a)
            r_t = torch.FloatTensor(r)
            ns_t = torch.FloatTensor(ns)
            d_t = torch.FloatTensor(d)
            q = agent(s_t).gather(1, a_t.unsqueeze(1)).squeeze()
            with torch.no_grad():
                nq = target_net(ns_t).max(1)[0]
            target = r_t + 0.99 * nq * (1-d_t)
            loss = nn.functional.mse_loss(q, target)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        if done: break
    rewards_log.append(ep_reward)
    EPSILON = max(0.05, EPSILON * 0.995)
    if episode % 50 == 0:
        target_net.load_state_dict(agent.state_dict())

window = 20
smoothed = np.convolve(rewards_log, np.ones(window)/window, mode='valid')
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(smoothed, color='steelblue', linewidth=2)
ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward')
ax.set_title('DQN Recommender Learning Curve')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Final 50-ep mean reward: {np.mean(rewards_log[-50:]):.2f}')

## ✅ Key Takeaways1. Recommendations are naturally sequential — MDPs, not static classifiers2. The reward function design is critical: what counts as 'good' recommendation?3. Exploration matters: a pure exploit agent never discovers new genres the user might like4. The state must encode enough history to distinguish similar user contexts